In [6]:
import torch
import numpy as np
import os
import h5py
import hdf5plugin  # Ensures compatibility with compressed hdf5 ECGs
from lightning_modules import SupervisedTask
from models.ecg_models import ECGandPriorLVEFtoLabel

DEVICE = 'cuda:2'  # Update based on your system hardware

In [7]:
def load_pulse_hf_model(
    ckpt: str = '/storage2/shared/PULSE-HF/models/12_lead/checkpoints/best.ckpt', 
    device: str = DEVICE
) -> torch.nn.Module:
    """
    Load the trained PULSE-HF model from checkpoint.

    Parameters:
    - ckpt (str): Path to the PyTorch Lightning model checkpoint.
    - device (str): Device to map the model (CPU or CUDA).

    Returns:
    - model (torch.nn.Module): Loaded PULSE-HF model ready for inference.
    """
    lightning_module = SupervisedTask.load_from_checkpoint(ckpt, map_location=torch.device(device))
    model = lightning_module.model
    return model

In [8]:
def preprocess_mgb_ecg(filepath: str, timestamp: str) -> torch.Tensor:
    """
    Load and preprocess a raw 12-lead ECG from an MGH hdf5 file into a fixed-size tensor.

    Parameters:
    - filepath (str): Path to the `.hd5` ECG file.
    - timestamp (str): ECG acquisition datetime, e.g., '2008-02-18 22:05:53'.
    - samples (int): Number of time-domain samples to resample each lead to.

    Returns:
    - ecg_tensor (torch.Tensor): A 12x2500 tensor (12 leads, each 2500-length).
    """
    ecg_key = str(timestamp).replace(' ', 'T')
    samples = 2500
    leads = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
    ecg_group = h5py.File(filepath, 'r')['ecg']

    if ecg_key not in ecg_group:
        raise KeyError(f"Timestamp '{ecg_key}' not found in file. Available keys: {list(ecg_group.keys())}")

    ecg_data_raw = ecg_group[ecg_key]

    ecg_signals = []
    for lead in leads:
        signal = ecg_data_raw[lead][()].reshape(-1)
        # Resample to standard length
        signal = np.interp(np.linspace(0, 1, samples), np.linspace(0, 1, len(signal)), signal)
        # Convert from microvolts to millivolts
        signal = signal / 1000
        ecg_signals.append(signal)

    ecg_array = np.vstack(ecg_signals)
    if np.isnan(ecg_array).any():
        raise ValueError(f"NaNs found in ECG signal from file: {filepath}")

    ecg_tensor = torch.tensor(ecg_array, dtype=torch.float64)  # Shape: [12, 2500]
    assert torch.isfinite(ecg_tensor).all(), "ECG contains non-finite values"
    return ecg_tensor

ecg_date = '2008-02-18 22:05:53'
ecg_path = '/storage2/shared/ecg/mgh/{PATIENT_ID}.hd5' 
ecg_features = preprocess_mgb_ecg(ecg_path, ecg_date)

In [9]:
def preprocess_lvef(lvef_history, days_since_diagnosis, diagnosis_lvef): 
    """
    Generates a 15-dimensional feature vector from LVEF history for use in the PULSE–HF model.

    This function captures both temporal and statistical information about a patient’s
    left ventricular ejection fraction (LVEF) over the past year, along with diagnostic context.
    These features are designed to mirror those used in the published PULSE-HF model to
    predict future heart failure progression.

    Parameters:
    ----------
    lvef_history : list[float]
        List of previous LVEF values measured within one year before the index ECG.
    
    days_since_diagnosis : int
        Number of days between the heart failure diagnosis and the index ECG.
    
    diagnosis_lvef : float
        The initial LVEF measurement at the time of heart failure diagnosis.

    Returns:
    -------
    torch.Tensor
        A 15-length tensor containing the following features:
            [0]  days_since_diagnosis  
            [1]  initial_lvef (at diagnosis)
            [2]  preceding_lvef (most recent prior to ECG)
            [3]  prior_lvef_support (count)
            [4]  prior_lvef_mean  
            [5]  prior_lvef_min  
            [6]  prior_lvef_max  
            [7]  prior_lvef_std  
            [8]  prior_lvef_range  
            [9]  any LVEF ≤ 40% (HF with reduced EF)
            [10] all LVEF ≤ 40%
            [11] any LVEF between 40–50% (HF with mildly reduced EF)
            [12] all LVEF between 40–50%
            [13] any LVEF ≥ 50% (HF with preserved EF)
            [14] all LVEF ≥ 50%
    
    Raises:
    -------
    AssertionError:
        If the resulting feature tensor contains NaNs or infinite values.
    """
    initial_lvef = diagnosis_lvef
    preceding_lvef = lvef_history[-1]
    prior_lvef_support = len(lvef_history)
    if not prior_lvef_support: 
        prior_lvef_mean = 0
        prior_lvef_min = 0
        prior_lvef_max = 0
        prior_lvef_std = 0
        prior_lvef_range = 0
        prior_lvef_any_hfref = 0
        prior_lvef_any_hfpef = 0
        prior_lvef_any_hfmref = 0
        prior_lvef_all_hfref = 0
        prior_lvef_all_hfmref = 0
        prior_lvef_all_hfpef = 0
    else:
        prior_lvef_mean = np.mean(lvef_history)
        prior_lvef_min = np.min(lvef_history)
        prior_lvef_max = np.max(lvef_history)
        prior_lvef_std = np.std(lvef_history)
        prior_lvef_range = prior_lvef_max - prior_lvef_min
        prior_lvef_any_hfref = int(np.any([x <= 40 for x in lvef_history]))
        prior_lvef_any_hfpef = int(np.any([40 < x < 50 for x in lvef_history]))
        prior_lvef_any_hfmref = int(np.any([x >= 50 for x in lvef_history]))
        prior_lvef_all_hfref = int(np.all([x <= 40 for x in lvef_history]))
        prior_lvef_all_hfmref = int(np.all([40 < x < 50 for x in lvef_history]))
        prior_lvef_all_hfpef = int(np.all([x >= 50 for x in lvef_history]))

    features = [days_since_diagnosis, initial_lvef, preceding_lvef, prior_lvef_support, prior_lvef_mean, prior_lvef_min, prior_lvef_max, prior_lvef_std, prior_lvef_range, prior_lvef_any_hfref, prior_lvef_all_hfref, prior_lvef_any_hfpef, prior_lvef_all_hfpef, prior_lvef_any_hfmref, prior_lvef_all_hfmref]
    features = torch.tensor(features, dtype=torch.float64)
    assert torch.isfinite(features).all(), f'nans:{features.isnan().sum()} infs:{features.isinf().sum()}'
    return features

lvef_features = preprocess_lvef(lvef_history=[25,47,54,37], days_since_diagnosis=23, diagnosis_lvef=22)

In [10]:
def get_pulse_hf_ecg_embedding(
    ecg: torch.Tensor,
    device: str = DEVICE
) -> torch.Tensor:
    """
    Extracts the deep ECG embedding from the PULSE–HF model.

    This embedding represents a learned representation of cardiac electrical activity,
    extracted using a ResNet-based encoder and decoder as described in the paper. The 
    output can be used for downstream tasks (prediction, visualization, etc.).

    Parameters:
    - ecg (torch.Tensor): A 2D tensor of shape [12, 2500] representing the 12-lead ECG.
    - device (str): Device identifier for inference ('cpu' or 'cuda').

    Returns:
    - ecg_embedding (torch.Tensor): A 1D tensor (i.e., latent embedding) representing the ECG.
    """
    assert ecg.shape == (12, 2500), f"Expected ECG shape [12, 2500], got {ecg.shape}"
    assert torch.isfinite(ecg).all(), "ECG signal contains NaNs or Infs"

    model = load_pulse_hf_model(device=device)
    model.to(device)
    model.eval()

    ecg = ecg.unsqueeze(0).float().to(device)  # Shape: [1, 12, 2500]

    with torch.no_grad():
        ecg_features = model.ecg_encoder(ecg)      # Intermediate features, ResNet [256 dim repr]
        ecg_embedding = model.ecg_decoder(ecg_features)  # Latent ECG representation, MLP

    return ecg_embedding.squeeze()  # Shape: [512]

get_pulse_hf_ecg_embedding(ecg=ecg_features).shape

/home/chandak/miniconda3/envs/ecg/lib/python3.8/site-packages/lightning/fabric/utilities/cloud_io.py:55: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


torch.Size([512])

In [11]:
def get_pulse_hf_score(
    ecg: torch.Tensor,
    lvef: torch.Tensor,
    device: str = DEVICE
) -> float:
    """
    Perform inference using the PULSE-HF model and return a probability score.

    This function computes the predicted probability that a patient’s LVEF
    will be below 40% within one year of the index ECG, based on combined
    ECG and LVEF history features, as described in the PULSE–HF paper.

    Parameters:
    ----------
    ecg : torch.Tensor
        A 2D tensor of shape [12, 2500] representing the raw ECG signal.

    lvef : torch.Tensor
        A 1D tensor of shape [15] containing engineered LVEF history features.

    device : str
        Device to perform inference on ("cpu" or "cuda").

    Returns:
    -------
    float
        A probability value (between 0 and 1) indicating predicted risk 
        of LVEF dropping below 40% within a year.
    """
    model = load_pulse_hf_model(device=device)
    model.to(device)
    model.eval()

    ecg = ecg.unsqueeze(0).float().to(device)   # Shape: [1, 12, 2500]
    lvef = lvef.unsqueeze(0).float().to(device) # Shape: [1, 15]

    with torch.no_grad():
        ecg_embed = model.ecg_decoder(model.ecg_encoder(ecg))
        lvef_embed = model.prior_lvef_decoder(lvef)
        combined = ecg_embed + lvef_embed
        logits = model.mlp(combined).squeeze()
        score = torch.sigmoid(logits)  # Apply sigmoid for probability

    return score.item()

get_pulse_hf_score(ecg_features, lvef_features)

0.6804901957511902